# Working with raw MultispeQ traces

Most datasets include a `sample_raw` column containing the **full raw measurement trace** from the MultispeQ device, stored as a Databricks VARIANT type. This column is excluded by the default `src.data` loaders because VARIANT is not supported by Databricks Connect / pandas.

This notebook shows how to access and parse the raw traces directly via PySpark SQL.

**What's in `sample_raw`?**
- Time-resolved fluorescence signals (ambient and high actinic light)
- Absorbance traces (820nm for PSI)
- PIRK (Post-Illumination Rise in fluorescence Kinetics)
- Environmental sensor readings at each time point
- Autogain calibration data

**Datasets with `sample_raw`:** all except Cowpea IITA (08).

**Requires:** Databricks (PySpark with VARIANT support).

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.connect.session import SparkSession

CATALOG = "open_jii_data_hackathon.default"

spark = SparkSession.getActiveSession()
assert spark is not None, "This notebook must be run on Databricks"

## Inspecting the raw trace structure

The `sample_raw` column is a JSON object stored as VARIANT. Let's look at one record to understand its structure.

In [ ]:
# Load just the raw column for one measurement
one = spark.sql(f"""
    SELECT measurement_id, sample_raw
    FROM {CATALOG}.grebbedijk_measurements
    LIMIT 1
""")

one.printSchema()

In [ ]:
# List all top-level keys in the raw trace
keys = spark.sql(f"""
    SELECT DISTINCT v.key
    FROM {CATALOG}.grebbedijk_measurements,
    LATERAL VARIANT_EXPLODE_OUTER(sample_raw) AS v
    WHERE v.key IS NOT NULL
    ORDER BY v.key
""")

keys.show(100, truncate=False)

## Extracting fluorescence traces

The raw JSON contains arrays of fluorescence readings at each time point. These can be extracted with `variant_get` or the `:` path syntax.

In [ ]:
# Extract fluorescence trace arrays from the first 5 measurements
traces = spark.sql(f"""
    SELECT
        measurement_id,
        sample_raw:data_raw AS data_raw
    FROM {CATALOG}.grebbedijk_measurements
    LIMIT 5
""")

traces.show(truncate=80)

In [ ]:
# Explode a trace array into individual time points
# This gives one row per (measurement, time_index, signal_value)
exploded = spark.sql(f"""
    SELECT
        measurement_id,
        POSEXPLODE(
            FROM_JSON(
                CAST(sample_raw:data_raw AS STRING),
                'ARRAY<DOUBLE>'
            )
        ) AS (time_index, signal)
    FROM {CATALOG}.grebbedijk_measurements
    LIMIT 5
""")

print(f"Rows: {exploded.count()}")
exploded.show(20)

## Batch extraction: custom phenotypes from raw data

You can extract specific fields from `sample_raw` across all measurements to compute your own phenotypes.

In [ ]:
# Example: extract specific nested values from all measurements
custom = spark.sql(f"""
    SELECT
        measurement_id,
        CAST(sample_raw:light_intensity AS DOUBLE) AS light_intensity,
        CAST(sample_raw:temperature AS DOUBLE) AS temperature,
        CAST(sample_raw:humidity AS DOUBLE) AS humidity
    FROM {CATALOG}.grebbedijk_measurements
    WHERE sample_raw IS NOT NULL
    LIMIT 20
""")

custom.show()

## Accessing other datasets

The same approach works for any dataset with `sample_raw`. Just change the table name:

| Table | Protocol | Notes |
| --- | --- | --- |
| `grebbedijk_measurements` | UNZA_PIRK_DIRK | Potato, includes PIRK traces |
| `bean_gart_35462_35509` | UNZA_PIRK_DIRK | Bean drought stress |
| `barley_qtl_32593_32742` | UNZA_PIRK_DIRK | Barley QTL mapping |
| `aardaker_nergena_29273` | RIDES 2.0 | Full ECS/PSI/P700 traces |
| `barley_imagic_17237_18685` | RIDES 2.0 | Ethiopia multi-location |
| `barley_hvdrr_12922_16934` | RIDES + PSII | Germany, two protocols |

In [ ]:
# Example: check what keys are available in a RIDES 2.0 dataset
rides_keys = spark.sql(f"""
    SELECT DISTINCT v.key
    FROM {CATALOG}.aardaker_nergena_29273,
    LATERAL VARIANT_EXPLODE_OUTER(sample_raw) AS v
    WHERE v.key IS NOT NULL
    ORDER BY v.key
""")

rides_keys.show(100, truncate=False)